# Agentic ML classification — end-to-end pipeline

This notebook walks through the full pipeline cell by cell, using the
same functions the CLI scripts (`scripts/run_orchestrator.py` etc.)
call — nothing here is reimplemented, it's the same
`agentic_ml.steps.*` / `agentic_ml.harness.*` code, just run one phase
at a time so you can see the intermediate output.

Pipeline: **intake** (optional — infers the target column from a
natural-language goal or the schema alone) → **profiler** (deterministic
facts + LLM narrative; its `recommended_split_strategy` drives the
split) → **split + leakage checks** → **modeling agent** (tries a few
candidate template+config proposals) → **select the best candidate
that passes the leakage gate** → **one locked test-set evaluation** →
**narrated summary**.

Requires a working `.env` (copy from `.env.example`) with `RIT_API_KEY`
/ `RIT_BASE_URL` set, or `MODEL_GATEWAY_BASE_URL` if using the gateway.
The API key is loaded but never printed anywhere in this notebook.

## 0. Setup — locate the repo, load `.env`, import everything

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "agentic_ml").exists():
            return candidate
    raise RuntimeError("Could not locate repo root (expected to find src/agentic_ml/ above cwd)")


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)  # so relative paths (datasets/raw/..., runs/, artifacts/...) match the CLI scripts
print(f"Repo root: {REPO_ROOT}")

In [ ]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        print(f"No .env file found at {path} — assuming environment variables are already set.")
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))


load_env_file(REPO_ROOT / ".env")
assert os.environ.get("RIT_API_KEY") or os.environ.get("MODEL_GATEWAY_BASE_URL"), (
    "No RIT_API_KEY or MODEL_GATEWAY_BASE_URL found. Copy .env.example to .env and fill in "
    "RIT_API_KEY / RIT_BASE_URL (or the gateway vars) before running this notebook."
)
print("Environment loaded (secrets not printed).")

In [ ]:
from sklearn.base import clone
from sklearn.metrics import auc, confusion_matrix, ConfusionMatrixDisplay, roc_curve
import matplotlib.pyplot as plt

from agentic_ml.cli_common import make_run_dir, make_tracer, resolve_model_endpoint
from agentic_ml.harness.dataset import DatasetSpec, load_dataset, read_dataframe
from agentic_ml.harness.leaderboard import append_leaderboard_entry
from agentic_ml.harness.leakage import run_all_split_leakage_checks
from agentic_ml.harness.metrics import compute_metrics
from agentic_ml.harness.splits import make_split
from agentic_ml.model_client import ModelClient
from agentic_ml.steps.intake_step import run_intake_step
from agentic_ml.steps.modeling_step import run_modeling_step
from agentic_ml.steps.profiler_step import run_profiler_step

print("Imports OK.")

## 1. Configuration

Edit this cell for your dataset, then run everything below it in order.

- Leave `target_column` as `None` to let the **intake agent** infer it
  from `goal` (or from the schema alone if `goal` is also empty) — the
  "just a dataset" case.
- Set `target_column` explicitly to skip intake entirely.

In [ ]:
CONFIG = {
    "data_path": "datasets/raw/train.csv",
    "goal": "predict whether a passenger survived the Titanic disaster",
    "target_column": None,        # e.g. "Survived" to skip intake
    "group_column": None,
    "time_column": None,
    "id_columns": None,           # comma-separated string, or None to let intake decide
    "strategy": None,             # None = use the profiler's recommendation
    "seed": 42,
    "metrics": ["roc_auc", "pr_auc", "f1", "accuracy"],
    "max_candidates": 2,
    "model": None,                # None = RIT_DEFAULT_MODEL env var
    "use_gateway": False,
    "run_id": None,               # None = auto-generate
}
CONFIG

## 2. Model client + run directory

In [ ]:
base_url, api_key, default_model = resolve_model_endpoint(
    CONFIG["use_gateway"], CONFIG["model"], "qwen3-coder:30b", "rit-qwen3-coder-30b",
)
client = ModelClient(base_url=base_url, api_key=api_key, default_model=default_model)

run_id, run_dir = make_run_dir(CONFIG["run_id"])
trace = make_tracer(run_dir / "trace.jsonl")

print(f"run_id: {run_id}")
print(f"run_dir: {run_dir}")
print(f"model: {default_model}")

## 3. Intake — figure out the dataset spec

Skipped if `CONFIG["target_column"]` is set. Otherwise the intake agent
sees only pre-target column facts (`get_raw_schema` — no target-dependent
stats exist yet) plus the natural-language `goal`, and proposes a
target/id/group/time column split. Its proposal is re-validated against
those same facts before anything downstream runs.

In [ ]:
if CONFIG["target_column"] is None:
    raw_df = read_dataframe(CONFIG["data_path"])
    intake_result = run_intake_step(
        raw_df, CONFIG["goal"] or "", client, model=default_model,
        trace_fn=lambda r: trace(**r),
    )

    display(pd.DataFrame(intake_result.raw_schema["columns"]))
    print("IntakeAgent proposal:")
    print(json.dumps(intake_result.dataset_spec_proposal, indent=2))

    assert intake_result.ok, f"Intake proposal failed validation: {intake_result.validation_errors}"

    proposal = intake_result.dataset_spec_proposal
    target_column = proposal["target_column"]
    group_column = CONFIG["group_column"] or proposal.get("group_column")
    time_column = CONFIG["time_column"] or proposal.get("time_column")
    id_columns = proposal.get("id_columns") or []
else:
    target_column = CONFIG["target_column"]
    group_column = CONFIG["group_column"]
    time_column = CONFIG["time_column"]
    id_columns = [c.strip() for c in (CONFIG["id_columns"] or "").split(",") if c.strip()]

print(f"\ntarget_column={target_column!r}  group_column={group_column!r}  "
      f"time_column={time_column!r}  id_columns={id_columns}")

## 4. Load the dataset + Profiler agent

Deterministic code computes every fact (`harness/profiler.py`); the LLM
only narrates and recommends — it cannot change `recommended_split_strategy`
or the leakage risk flags.

In [ ]:
spec = DatasetSpec(
    path=CONFIG["data_path"], target_column=target_column,
    group_column=group_column, time_column=time_column, id_columns=id_columns,
)
loaded = load_dataset(spec)
display(loaded.df.head())
print(f"{len(loaded.df)} rows, {len(loaded.df.columns)} columns, data_hash={loaded.data_hash[:16]}...")

In [ ]:
profiler_result = run_profiler_step(
    loaded.df, target_column, client, model=default_model,
    trace_fn=lambda r: trace(**r),
)
assert profiler_result.ok, "ProfilerAgent never called get_dataset_profile."

dr = profiler_result.deterministic_report
display(pd.DataFrame(dr["columns"]))
print(f"recommended_split_strategy: {dr['recommended_split_strategy']}")
print(f"is_imbalanced: {dr['is_imbalanced']}  (ratio={dr['class_imbalance_ratio']})")
print(f"leakage_risk_flags: {dr['leakage_risk_flags']}")

display(Markdown(f"**Profiler narrative:**\n\n{profiler_result.llm_narrative.get('summary', '') if profiler_result.llm_narrative else profiler_result.llm_raw_text}"))

## 5. Split + split-level leakage checks

In [ ]:
strategy = CONFIG["strategy"] or dr["recommended_split_strategy"]
print(f"Split strategy: {strategy} ({'explicit override' if CONFIG['strategy'] else 'profiler recommendation'})")

manifest = make_split(
    df=loaded.df, target_column=target_column, data_hash=loaded.data_hash,
    strategy=strategy, seed=CONFIG["seed"], group_column=group_column, time_column=time_column,
)
print(f"train={len(manifest.train_idx)}  val={len(manifest.val_idx)}  test={len(manifest.test_idx)}")
display(pd.DataFrame(manifest.target_distribution))

In [ ]:
leakage_checks = run_all_split_leakage_checks(
    df=loaded.df, group_column=group_column, time_column=time_column,
    train_idx=manifest.train_idx, val_idx=manifest.val_idx, test_idx=manifest.test_idx,
    strategy=strategy,
)
leakage_df = pd.DataFrame([c.to_dict() for c in leakage_checks])
display(leakage_df)

assert all(c.passed for c in leakage_checks), "Split-level leakage check failed — stopping here."
print("All split-level leakage checks passed.")

## 6. Modeling agent — propose & evaluate candidates

Each candidate: the agent picks a verified recipe template and fills in
its config (feature columns, a few hyperparameters) — it never writes
pipeline code. The harness then re-validates every proposed column
against the profiler's facts, static-checks + sandbox-builds the
template, fits/scores it on the train/val split, and runs a
label-permutation leakage gate. `ok=True` means "passed the gate", not
just "no errors".

In [ ]:
metric_names = CONFIG["metrics"]
primary_metric = metric_names[0]
candidate_results = []
tried_template_ids = []
leaderboard_path = Path("artifacts/reports/leaderboard.jsonl")

for i in range(CONFIG["max_candidates"]):
    print(f"\n=== ModelingAgent candidate {i + 1}/{CONFIG['max_candidates']} ===")
    step_result = run_modeling_step(
        full_df=loaded.df, X=loaded.X, y=loaded.y, target_column=target_column,
        group_column=group_column, time_column=time_column,
        train_idx=manifest.train_idx, val_idx=manifest.val_idx,
        client=client, model=default_model, metric_names=metric_names, seed=CONFIG["seed"],
        already_tried_template_ids=tried_template_ids,
        trace_fn=lambda r: trace(**r),
    )
    if step_result.template_id:
        tried_template_ids.append(step_result.template_id)
    candidate_results.append(step_result)

    print(f"candidate_id={step_result.candidate_id}  template={step_result.template_id}")
    if step_result.explanation:
        print(f"explanation: {step_result.explanation}")

    if step_result.ok:
        print(f"PASSED leakage gate — {primary_metric}={step_result.metrics[primary_metric]['value']:.4f}")
        append_leaderboard_entry(leaderboard_path, {
            "run_id": run_id, "candidate": step_result.candidate_id,
            "template_id": step_result.template_id, "source": "notebook",
            "model": default_model, "split": "validation", "strategy": strategy,
            "data_hash": loaded.data_hash, "seed": CONFIG["seed"], "metrics": step_result.metrics,
        })
    else:
        print(f"REJECTED: {step_result.errors}")

rows = []
for r in candidate_results:
    row = {"candidate_id": r.candidate_id, "template_id": r.template_id, "passed_gate": r.ok}
    if r.metrics:
        row.update({m: v["value"] for m, v in r.metrics.items()})
    rows.append(row)

comparison_df = pd.DataFrame(rows)
display(comparison_df)

## 7. Select the best candidate + final test-set evaluation

The winner (by validation `primary_metric`, among candidates that
passed the leakage gate) is refit on train+val combined, then the
**locked test set is touched exactly once**.

In [ ]:
passing = [r for r in candidate_results if r.ok]
assert passing, "No candidate passed the label-permutation leakage gate — nothing to promote."

best = max(passing, key=lambda r: r.metrics[primary_metric]["value"])
print(f"Selected: {best.candidate_id} ({best.template_id}) — "
      f"{primary_metric}={best.metrics[primary_metric]['value']:.4f} on validation")

train_and_val_idx = sorted(manifest.train_idx + manifest.val_idx)
final_pipeline = clone(best.pipeline)
final_pipeline.fit(loaded.X.iloc[train_and_val_idx], loaded.y.iloc[train_and_val_idx])

y_test = loaded.y.iloc[manifest.test_idx]
y_pred = final_pipeline.predict(loaded.X.iloc[manifest.test_idx])
proba = final_pipeline.predict_proba(loaded.X.iloc[manifest.test_idx])
y_proba = proba[:, 1] if proba.shape[1] == 2 else proba.max(axis=1)

test_results = compute_metrics(y_test.values, y_pred, y_proba, metric_names, n_bootstrap=200, seed=CONFIG["seed"])
test_metrics_df = pd.DataFrame([r.to_dict() for r in test_results.values()]).set_index("metric")
display(test_metrics_df)
print("\nNOTE: this is the only time the test split is touched.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_value = auc(fpr, tpr)
axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc_value:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve (test set)")
axes[0].legend()

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Confusion matrix (test set)")

plt.tight_layout()
plt.show()

## 8. Narrated summary

Plain-text only — no tools, no decisions, just narrating facts already computed above.

In [ ]:
summary_messages = [
    {"role": "system", "content": (
        "You are the Analyst step of a deterministic ML pipeline. Respond "
        "with 4-6 sentences of plain prose only (no JSON, no markdown "
        "fences) summarizing the given facts for a non-technical "
        "stakeholder. Do not invent any numbers not present in the input."
    )},
    {"role": "user", "content": json.dumps({
        "goal": CONFIG["goal"] or "(none given — target inferred from data alone)",
        "target_column": target_column,
        "n_candidates_tried": len(candidate_results),
        "n_candidates_passed_leakage_gate": len(passing),
        "selected_candidate": {"candidate_id": best.candidate_id, "template_id": best.template_id},
        "final_test_metrics": {m: test_results[m].to_dict() for m in metric_names},
    }, indent=2)},
]
summary_response = client.call(summary_messages, model=default_model, max_tokens=400)
display(Markdown(f"**Summary:**\n\n{summary_response.text}"))

## 9. Write a report + list this run's artifacts

In [ ]:
report = {
    "run_id": run_id, "model": default_model, "status": "success",
    "target_column": target_column,
    "selected_candidate": {"candidate_id": best.candidate_id, "template_id": best.template_id},
    "final_test_metrics": {m: test_results[m].to_dict() for m in metric_names},
    "final_summary": summary_response.text,
}
(run_dir / "notebook_report.json").write_text(json.dumps(report, indent=2, default=str))
print(f"Report written to {run_dir / 'notebook_report.json'}")

print("\nFiles written this run:")
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        print(" ", p)  # already relative — we chdir'd to REPO_ROOT in cell 0